# F# Computation Expressions

## Introduction

Most languages give you a fixed set of control flow primitives — `if`, `for`, `while`, `try/catch` — and that's that. F# has all of those, but it also lets you *define your own*. Computation expressions are F#'s mechanism for building custom syntax that can wrap and sequence operations in ways that feel native to the language.

The most familiar example from other languages is `async/await`. In C# or JavaScript, the compiler transforms `await` expressions into continuation-passing state machines behind the scenes. F# generalizes that idea: computation expressions give you the same power, but for any workflow you want to model — not just async code.

In practice, they shine brightest in three places:

- **`option`** — propagating the absence of a value through a chain of operations without a pyramid of `match` expressions
- **`result`** — threading errors through a pipeline so that the first failure short-circuits the rest
- **`async`** — composing asynchronous operations without explicit callbacks or `.ContinueWith` chains

In each case, the computation expression handles the plumbing — the "did this step succeed? okay, pass it to the next one" logic — so your code can focus on the happy path. The sections below walk through each one, showing what the code looks like before and after.

There are a couple of ways to handle this without a computation expression, and both have trade-offs.

> **Note:** The `option` computation expression is part of the F# standard library — no extra packages needed.

In [ ]:
// Imagine a small domain: users have an active subscription, subscriptions have a promo code
type PromoCode = { Code: string; DiscountPct: int }
type Subscription = { Plan: string; PromoCode: PromoCode option }
type User = { Name: string; Subscription: Subscription option }

let findUser (id: int) : User option =
    if id = 1 then Some { Name = "Alice"; Subscription = Some { Plan = "Pro"; PromoCode = Some { Code = "SAVE20"; DiscountPct = 20 } } }
    else None

// --- Before: cascading match expressions ---
// Each optional hop requires its own match arm. Nesting grows with the chain.
let getDiscountNestedMatch (userId: int) : int option =
    match findUser userId with
    | None -> None
    | Some user ->
        match user.Subscription with
        | None -> None
        | Some sub ->
            match sub.PromoCode with
            | None -> None
            | Some promo -> Some promo.DiscountPct

printfn "Nested match: %A" (getDiscountNestedMatch 1)   // Some 20
printfn "Nested match: %A" (getDiscountNestedMatch 99)  // None

// --- Before: Option.bind ---
// Option.bind f opt evaluates to: match opt with None -> None | Some x -> f x
// Piping through Option.bind flattens the nesting, but the lambda noise adds up.
let getDiscountOptionBind (userId: int) : int option =
    findUser userId
    |> Option.bind  (fun user  -> user.Subscription)
    |> Option.bind  (fun sub   -> sub.PromoCode)
    |> Option.map   (fun promo -> promo.DiscountPct)

printfn "Option.bind:  %A" (getDiscountOptionBind 1)   // Some 20
printfn "Option.bind:  %A" (getDiscountOptionBind 99)  // None

Nested match: Some 20
Nested match: None
Option.bind:  Some 20
Option.bind:  None


In [ ]:
// --- With the option computation expression ---
// `let!` unwraps a `Some`, or short-circuits with `None` if the value is absent.
// The happy path reads top-to-bottom with no nesting.
let getDiscountWithCE (userId: int) : int option =
    option {
        let! user  = findUser userId
        let! sub   = user.Subscription
        let! promo = sub.PromoCode
        return promo.DiscountPct
    }

printfn "option CE:    %A" (getDiscountWithCE 1)   // Some 20
printfn "option CE:    %A" (getDiscountWithCE 99)  // None

The logic is identical across all three, but each approach makes a different trade-off. Nested `match` is maximally explicit but creates a rightward drift that gets worse with every hop. `Option.bind` flattens the structure using a pipeline, which is better — but you're still writing a lambda for each step, and the ceremony starts to obscure what you're actually computing. The computation expression eliminates both problems: `let!` unwraps each option and short-circuits on `None`, while the block reads straight down like ordinary sequential code.

This pattern scales cleanly. Whether you're unwrapping two optional values or ten, the structure stays flat.

## Results

The `result` type takes the `option` idea further: instead of just signalling absence with `None`, it carries an error value when something goes wrong. A result is either `Ok value` or `Error err`, and like `option`, the type system forces you to handle both cases.

Chaining fallible operations has the same structural problem as chaining optional ones — each step might fail, and you want to short-circuit on the first error. The `result` CE from `FsToolkit.ErrorHandling` handles this cleanly.

> **Note:** Unlike `option`, the `result` CE is not in the F# standard library. It comes from [`FsToolkit.ErrorHandling`](https://github.com/demystifyfp/FsToolkit.ErrorHandling), a widely-used community package that adds computation expression support for `Result`, `Async`, and more.

In [ ]:
// A small validation pipeline: parse an order from raw input
type OrderError = InvalidId | NegativeQuantity | UnknownProduct
type Order = { Id: int; ProductId: int; Quantity: int }

let parseId (raw: string) : Result<int, OrderError> =
    match System.Int32.TryParse raw with
    | true, n when n > 0 -> Ok n
    | _ -> Error InvalidId

let validateQuantity (qty: int) : Result<int, OrderError> =
    if qty > 0 then Ok qty else Error NegativeQuantity

let lookupProduct (productId: int) : Result<int, OrderError> =
    if productId = 42 then Ok productId else Error UnknownProduct

// --- Before: cascading match expressions ---
let parseOrderNestedMatch (rawId: string) (qty: int) (productId: int) : Result<Order, OrderError> =
    match parseId rawId with
    | Error e -> Error e
    | Ok id ->
        match validateQuantity qty with
        | Error e -> Error e
        | Ok validQty ->
            match lookupProduct productId with
            | Error e -> Error e
            | Ok pid -> Ok { Id = id; ProductId = pid; Quantity = validQty }

printfn "Nested match: %A" (parseOrderNestedMatch "7" 3 42)   // Ok ...
printfn "Nested match: %A" (parseOrderNestedMatch "7" -1 42)  // Error NegativeQuantity

// --- Before: Result.bind ---
// Flattens the nesting but requires threading intermediate values manually.
let parseOrderResultBind (rawId: string) (qty: int) (productId: int) : Result<Order, OrderError> =
    parseId rawId
    |> Result.bind (fun id ->
        validateQuantity qty
        |> Result.bind (fun validQty ->
            lookupProduct productId
            |> Result.map (fun pid -> { Id = id; ProductId = pid; Quantity = validQty })))

printfn "Result.bind:  %A" (parseOrderResultBind "7" 3 42)   // Ok ...
printfn "Result.bind:  %A" (parseOrderResultBind "abc" 3 42) // Error InvalidId

In [ ]:
#r "nuget: FsToolkit.ErrorHandling"

open FsToolkit.ErrorHandling

Installed Packages FsToolkit.ErrorHandling, 5.2.0

In [ ]:
// --- With the result computation expression (FsToolkit.ErrorHandling) ---
// `let!` unwraps Ok, or short-circuits with the Error value.
// All intermediate values stay in scope — no manual threading.
let parseOrderCE (rawId: string) (qty: int) (productId: int) : Result<Order, OrderError> =
    result {
        let! id       = parseId rawId
        let! validQty = validateQuantity qty
        let! pid      = lookupProduct productId
        return { Id = id; ProductId = pid; Quantity = validQty }
    }

printfn "result CE: %A" (parseOrderCE "7" 3 42)    // Ok { Id = 7; ProductId = 42; Quantity = 3 }
printfn "result CE: %A" (parseOrderCE "abc" 3 42)  // Error InvalidId
printfn "result CE: %A" (parseOrderCE "7" 3 99)    // Error UnknownProduct

The `Result.bind` version avoids the rightward drift of nested matches, but notice that threading `id` and `validQty` into the final `Result.map` requires nesting the lambdas anyway — you can't easily keep all the bound values in scope when chaining with `|>`. The `result { }` CE solves both problems at once: it's flat, and every `let!` binding is naturally in scope for the rest of the block.

## Query Expressions

Computation expressions aren't limited to error handling. F# also has a built-in `query { }` CE that brings LINQ-style querying directly into the language — no extension methods, no lambda-heavy syntax.

```fsharp
let highValueOrders =
    query {
        for order in orders do
        where (order.Total > 100.0)
        sortByDescending order.Total
        select order
    }
```

This is not special syntax baked into the compiler — `query` is just another computation expression builder, defined in the standard library. `for`, `where`, `sortByDescending`, and `select` are all methods on the builder that the compiler desugars into `IQueryable`-compatible calls. The same mechanism that powers `option { }` and `result { }` powers SQL-translatable database queries.

That's the deeper point: computation expressions are a general-purpose abstraction. Once you understand the pattern — a builder type with `Bind`, `Return`, and friends — you can read (and write) any CE in the ecosystem, whether it's for async workflows, error handling, query composition, or something domain-specific you've built yourself.